# Evaluación Segunda Unidad IA – CNN sobre CIFAR-10
**Código de matrícula:** 022100575k → **semilla = 75**

Este notebook contiene: carga con semilla, EDA, visualización de muestras, pipeline de transformaciones, partición estratificada, validación de shapes 4D y una corrida de entrenamiento.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, torch
import matplotlib.pyplot as plt
%matplotlib inline
from dataset import SEED, CLASSES, set_seed, get_transforms, get_dataloaders, visualize_samples, stratified_split
from model import CNN, validate_forward_pass
set_seed(SEED)
print('Semilla:', SEED)

## 1. Carga, partición estratificada y dimensiones

In [ ]:
train_loader, val_loader, test_loader, (tr_idx, va_idx) = get_dataloaders(batch_size=128, num_workers=0)

## 2. EDA: distribución de clases por partición

In [ ]:
from torchvision import datasets
raw = datasets.CIFAR10('../data', train=True, download=True)
t = np.array(raw.targets)
fig, ax = plt.subplots(figsize=(9,3.5))
w = 0.4; x = np.arange(10)
ax.bar(x-w/2, np.bincount(t[tr_idx], minlength=10), w, label='Train')
ax.bar(x+w/2, np.bincount(t[va_idx], minlength=10), w, label='Validación')
ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=45); ax.set_title('Imágenes por clase (partición estratificada)'); ax.legend()
plt.tight_layout(); plt.show()
print('Pixel medio por canal RGB:', (raw.data/255.).mean(axis=(0,1,2)).round(4))
print('Desv. estándar por canal RGB:', (raw.data/255.).std(axis=(0,1,2)).round(4))

## 3. Visualización de muestras con su etiqueta real

In [ ]:
import dataset
dataset.matplotlib.use('module://matplotlib_inline.backend_inline')
fig = visualize_samples(16)
plt.show()

## 4. Efecto del Data Augmentation (RandomCrop + HorizontalFlip)

In [ ]:
train_tf, eval_tf = get_transforms()
img, lab = raw[0]
fig, axes = plt.subplots(1, 6, figsize=(14, 2.6))
axes[0].imshow(img); axes[0].set_title('Original: '+CLASSES[lab]); axes[0].axis('off')
mean = torch.tensor([0.4914,0.4822,0.4465]).view(3,1,1); std = torch.tensor([0.2470,0.2435,0.2616]).view(3,1,1)
for a in axes[1:]:
    a.imshow((train_tf(img)*std+mean).clamp(0,1).permute(1,2,0)); a.set_title('Aumentada'); a.axis('off')
plt.tight_layout(); plt.show()

## 5. Validación del forward pass con tensores 4D

In [ ]:
validate_forward_pass(batch_size=8)

In [ ]:
xb, yb = next(iter(train_loader))
model = CNN()
model.eval()
with torch.no_grad():
    out = model(xb, verbose=True)
assert out.shape == (xb.size(0), 10)
print('Batch real OK:', tuple(xb.shape), '->', tuple(out.shape))

## 6. Entrenamiento
Ejecutar `python src/train.py` desde la raíz del repositorio genera `outputs/loss_accuracy.png` y `outputs/confusion_matrix.png`. Abajo se muestran los artefactos.

In [ ]:
from IPython.display import Image, display
for f in ['../outputs/loss_accuracy.png', '../outputs/confusion_matrix.png']:
    if os.path.exists(f): display(Image(filename=f))
    else: print('Aún no existe', f, '-> ejecute: python src/train.py')